
# Chapter 12 — End-to-End Forecasting Project
# Retail Sales Forecasting Case Study




## 1. Setup & Libraries

Description:
Load required libraries for forecasting models,
preprocessing, evaluation, and deployment simulation.

In [6]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.arima.model import ARIMA

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [7]:
# If using Google Colab:
# Upload retail_sales.csv manually

from google.colab import files
uploaded = files.upload()

# Load dataset
df = pd.read_csv("retail_sales.csv")

# Convert date column
df['date'] = pd.to_datetime(df['date'])

# Sort by date
df = df.sort_values(by='date')

print(df.head())

Saving retail_sales.csv to retail_sales (3).csv
        date  sales
0 2023-01-01    120
1 2023-02-01    135
2 2023-03-01    150
3 2023-04-01    160
4 2023-05-01    155


## 3. Data Cleaning

In [8]:
# Handle missing values and remove extreme outliers.

# Fill missing values
df['sales'] = df['sales'].ffill()

# IQR Outlier Removal
q1 = df['sales'].quantile(0.25)
q3 = df['sales'].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df = df[
    (df['sales'] >= lower_bound) &
    (df['sales'] <= upper_bound)
]

print("\nCleaned Data Shape:", df.shape)


Cleaned Data Shape: (24, 2)


## 4. Feature Engineering

In [9]:
# Create time-based, lag, and rolling features.

# Time Features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Weekend Indicator
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Lag Features
df['lag_1'] = df['sales'].shift(1)
df['lag_3'] = df['sales'].shift(3)

# Rolling Mean
df['rolling_mean_3'] = df['sales'].rolling(3).mean()

# Remove missing rows generated by lagging
df.dropna(inplace=True)

print("\nFeature Engineered Data:")
print(df.head())


Feature Engineered Data:
        date  sales  day_of_week  month  is_weekend  lag_1  lag_3  \
3 2023-04-01    160            5      4           1  150.0  120.0   
4 2023-05-01    155            0      5           0  160.0  135.0   
5 2023-06-01    170            3      6           0  155.0  150.0   
6 2023-07-01    200            5      7           1  170.0  160.0   
7 2023-08-01    210            1      8           0  200.0  155.0   

   rolling_mean_3  
3      148.333333  
4      155.000000  
5      161.666667  
6      175.000000  
7      193.333333  


## 5. Train-Test Split

In [10]:
# Split data into training and testing sets.

train_size = int(len(df) * 0.8)

train = df.iloc[:train_size]
test = df.iloc[train_size:]

# Selected Features
features = [
    'lag_1',
    'lag_3',
    'rolling_mean_3',
    'day_of_week',
    'month',
    'is_weekend'
]

X_train = train[features]
y_train = train['sales']

X_test = test[features]
y_test = test['sales']

print("\nTraining Size:", len(train))
print("Testing Size:", len(test))


Training Size: 16
Testing Size: 5


## 6. Model 1 — ARIMA

In [11]:

# Classical statistical forecasting model.

arima_train = train['sales']

arima_model = ARIMA(arima_train, order=(2, 1, 2))

arima_fit = arima_model.fit()

arima_pred = arima_fit.forecast(steps=len(test))

## 7. Model 2 — Random forest

In [12]:
# Machine learning forecasting model using engineered features.

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

## 8. Model 3 — LSTM

In [13]:
# Deep learning forecasting model for sequential learning.

# Reshape data for LSTM
X_train_lstm = np.array(X_train).reshape(
    (X_train.shape[0], 1, X_train.shape[1])
)

X_test_lstm = np.array(X_test).reshape(
    (X_test.shape[0], 1, X_test.shape[1])
)

# Build LSTM Model
lstm_model = Sequential()

lstm_model.add(
    LSTM(
        50,
        activation='relu',
        input_shape=(1, X_train.shape[1])
    )
)

lstm_model.add(Dense(1))

# Compile
lstm_model.compile(
    optimizer='adam',
    loss='mse'
)

# Train
lstm_model.fit(
    X_train_lstm,
    y_train,
    epochs=20,
    verbose=0
)

# Predict
lstm_pred = lstm_model.predict(X_test_lstm).flatten()



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step



# 9. Model Evaluation


Description:
Compare forecasting models using MAE and RMSE.

In [14]:
def evaluate(y_true, y_pred, name):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    print(f"\n{name}")
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")

    return mae


mae_arima = evaluate(y_test, arima_pred, "ARIMA")

mae_rf = evaluate(y_test, rf_pred, "Random Forest")

mae_lstm = evaluate(y_test, lstm_pred, "LSTM")





ARIMA
MAE  : 33.93
RMSE : 38.56

Random Forest
MAE  : 8.83
RMSE : 11.24

LSTM
MAE  : 214.06
RMSE : 214.92



## 10. Model Selection

Description:
Automatically select best forecasting model.

In [15]:


def select_best_model(mae_arima, mae_rf, mae_lstm):

    scores = {
        "ARIMA": mae_arima,
        "Random Forest": mae_rf,
        "LSTM": mae_lstm
    }

    best_model = min(scores, key=scores.get)

    return best_model, scores


best_model, scores = select_best_model(
    mae_arima,
    mae_rf,
    mae_lstm
)

print("\n==============================")
print("BEST MODEL SELECTION")
print("==============================")

print("Best Model:", best_model)

print("\nAll Scores:")
print(scores)



BEST MODEL SELECTION
Best Model: Random Forest

All Scores:
{'ARIMA': 33.9267281240522, 'Random Forest': 8.830000000000002, 'LSTM': 214.06076049804688}



## 11. Simple Deployment (API SIMULATION)


Description:
Simulate production forecasting API.

In [16]:


from flask import Flask, request, jsonify

app = Flask(__name__)

# Assume Random Forest selected
final_model = rf_model


@app.route('/predict', methods=['POST'])
def predict():

    data = request.json

    features = np.array(
        data['features']
    ).reshape(1, -1)

    prediction = final_model.predict(features)

    return jsonify({
        'forecast': float(prediction[0])
    })


# Uncomment when running locally
# app.run()


## 12. Business Decision Layer

Description:
Convert forecasts into business actions.

In [17]:
def business_decision(forecast):

    if forecast > 200:
        return "Increase Inventory"

    elif forecast < 150:
        return "Reduce Inventory"

    else:
        return "Maintain Current Inventory"


sample_forecast = rf_model.predict(
    X_test.iloc[:1]
)[0]

print("\n==============================")
print("BUSINESS DECISION ENGINE")
print("==============================")

print("Forecast Value:", round(sample_forecast, 2))

print(
    "Business Decision:",
    business_decision(sample_forecast)
)


BUSINESS DECISION ENGINE
Forecast Value: 207.9
Business Decision: Increase Inventory
